# NB 05 — Registros SUNAT de Compras y Ventas
**Proyecto 5 · Automatización Contable**

**Input:** `data/processed/facturas_validadas.json`

**Output:**
- `data/output/Registro_Compras_{periodo}.xlsx` — Formato SUNAT 8.1
- `data/output/Registro_Ventas_{periodo}.xlsx` — Formato SUNAT 14.1

No requiere plan de cuentas — rubro-agnóstico. Clasificación tributaria por regla determinista (gravada por defecto, excepciones a revisión manual).

## 0. Setup

In [1]:
import json
import openpyxl
from openpyxl.styles import PatternFill
from pathlib import Path
from datetime import datetime, date
from dotenv import load_dotenv
import cliente_config
from convertir_plantilla_sunat import convertir_si_falta

load_dotenv(dotenv_path=Path('D:/Proyecto_Gabriel/02_Agente_IA/Skill_financiero/.env'))

BASE_DIR    = Path('../')
CONFIG_DIR  = BASE_DIR / 'config'
INPUT_PATH  = BASE_DIR / 'data/processed/facturas_validadas.json'
OUTPUT_DIR  = BASE_DIR / 'data/output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLIENTE_ID = 'carlos_torres'
CLIENTE = cliente_config.cargar_cliente(CLIENTE_ID, CONFIG_DIR)

PLANTILLA_COMPRAS = convertir_si_falta(
    Path('D:/Proyecto_Gabriel/01_Portfolio/Proyecto_5_Demo/config/234_formato81_compra.xls'),
    CONFIG_DIR / 'planes/formato81_compra.xlsx'
)
PLANTILLA_VENTAS = convertir_si_falta(
    Path('D:/Proyecto_Gabriel/01_Portfolio/Proyecto_5_Demo/config/234_formato141_venta.xls'),
    CONFIG_DIR / 'planes/formato141_venta.xlsx'
)

FILL_AMARILLO = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')

print(f'Cliente activo: {CLIENTE["nombre"]}')
print(f'RUC: {CLIENTE.get("ruc")}  Razon social: {CLIENTE.get("razon_social")}')
print(f'Plantilla compras: {PLANTILLA_COMPRAS}')
print(f'Plantilla ventas: {PLANTILLA_VENTAS}')

Cliente activo: CT PRIME CONSULTING SAC (libros de Contacto Creativo TI S.A.C.)
RUC: 20608899473  Razon social: CONTACTO CREATIVO TI S.A.C.
Plantilla compras: ..\config\planes\formato81_compra.xlsx
Plantilla ventas: ..\config\planes\formato141_venta.xlsx


## 1. Helpers (fecha, redondeo, correlativo, dedup)

Nota: las plantillas oficiales solo traen ~10 filas en blanco entre el encabezado y la fila de TOTALES/notas al pie. Si un periodo real supera esa cantidad de comprobantes, escribir_fila_* sobreescribiría la fila de TOTALES — fuera de alcance por ahora (el piloto actual tiene 1-4 comprobantes por periodo).

In [2]:
def parse_fecha(fecha_str) -> datetime:
    if isinstance(fecha_str, datetime):
        return fecha_str
    if isinstance(fecha_str, date):
        return datetime(fecha_str.year, fecha_str.month, fecha_str.day)
    try:
        return datetime.strptime(str(fecha_str)[:10], '%Y-%m-%d')
    except Exception:
        return None


def round2(val) -> float:
    try:
        return round(float(val or 0), 2)
    except (ValueError, TypeError):
        return 0.0


def get_ultimo_correlativo(ws, fila_inicio_datos: int) -> int:
    maximo = 0
    for row in ws.iter_rows(min_row=fila_inicio_datos, values_only=True):
        val = row[0]
        if isinstance(val, (int, float)) and val > maximo:
            maximo = int(val)
    return maximo


def get_primera_fila_libre(ws, fila_inicio_datos: int) -> int:
    """Primera fila vacia (columna correlativo=None) a partir de fila_inicio_datos.
    NO usar ws.max_row: la plantilla ya trae filas de TOTALES y notas al pie
    mas abajo, que ws.max_row si cuenta."""
    fila = fila_inicio_datos
    while ws.cell(row=fila, column=1).value is not None:
        fila += 1
    return fila


def leer_docs_existentes(ws, fila_inicio_datos: int, col_serie: int, col_numero: int) -> set:
    existentes = set()
    for row in ws.iter_rows(min_row=fila_inicio_datos, values_only=True):
        serie = row[col_serie - 1]
        numero = row[col_numero - 1]
        if serie or numero:
            existentes.add(f'{serie}-{numero}')
    return existentes


def parsear_referencia(doc_referencia) -> tuple[str, str]:
    if not doc_referencia or '-' not in str(doc_referencia):
        return '', ''
    serie, numero = str(doc_referencia).split('-', 1)
    return serie.strip(), numero.strip()


def construir_lookup_referencias(facturas: list[dict]) -> dict:
    lookup = {}
    for f in facturas:
        clave = (str(f.get('serie', '')).strip(), str(f.get('numero', '')).strip())
        lookup[clave] = {
            'fecha_emision': f.get('fecha_emision'),
            'codigo_tipo_doc': f.get('codigo_tipo_doc', '01'),
        }
    return lookup

print('Helpers OK')

Helpers OK


## 2. Clasificación tributaria

In [3]:
TIPOS_DOC_COMUNES = {'FACTURA', 'BOLETA'}


def clasificar_tributario(factura: dict) -> dict:
    """
    Regla determinista: gravada por defecto si hay IGV y el comprobante es
    de un tipo comun (Factura/Boleta). NC/ND y cualquier otro caso quedan
    marcados para revision manual (no se adivina exonerada/inafecta/no gravada).
    """
    igv = float(factura.get('igv') or 0)
    tipo_doc = factura.get('tipo_doc', '')

    if igv > 0 and tipo_doc in TIPOS_DOC_COMUNES:
        return {'tratamiento_tributario': 'GRAVADA', 'requiere_revision_tributaria': False}

    return {'tratamiento_tributario': None, 'requiere_revision_tributaria': True}

print('Clasificacion tributaria lista')

Clasificacion tributaria lista


## 3. Registro de Compras (Formato 8.1)

In [4]:
def escribir_fila_compra(ws, fila: int, correlativo: int, factura: dict, lookup_ref: dict) -> None:
    tratamiento = factura.get('tratamiento_tributario')
    es_ajuste = factura.get('tipo_doc') in ('NOTA_CREDITO', 'NOTA_DEBITO')

    base = round2(factura.get('base_imponible'))
    igv = round2(factura.get('igv'))
    total = round2(factura.get('total'))
    # Normalizar: si extracción fue inconsistente, recalcular desde total con IGV 18%
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv = round2(total - base)

    ws.cell(row=fila, column=1, value=correlativo)
    ws.cell(row=fila, column=2, value=parse_fecha(factura.get('fecha_emision')))
    ws.cell(row=fila, column=3, value=parse_fecha(factura.get('fecha_vencimiento_pago')))
    ws.cell(row=fila, column=4, value=factura.get('codigo_tipo_doc'))
    ws.cell(row=fila, column=5, value=factura.get('serie'))
    # columna 6 (año DUA) queda vacia — fuera de alcance
    ws.cell(row=fila, column=7, value=factura.get('numero'))
    ws.cell(row=fila, column=8, value='6')  # proveedor siempre RUC
    ws.cell(row=fila, column=9, value=factura.get('ruc_emisor'))
    ws.cell(row=fila, column=10, value=factura.get('razon_social_emisor'))

    if tratamiento == 'GRAVADA':
        ws.cell(row=fila, column=11, value=base)
        ws.cell(row=fila, column=12, value=igv)

    ws.cell(row=fila, column=20, value=total)

    if factura.get('moneda') == 'USD' and factura.get('tipo_cambio'):
        ws.cell(row=fila, column=24, value=factura.get('tipo_cambio'))

    if es_ajuste:
        ref_serie, ref_numero = parsear_referencia(factura.get('doc_referencia'))
        ref_info = lookup_ref.get((ref_serie, ref_numero), {})
        ws.cell(row=fila, column=25, value=parse_fecha(ref_info.get('fecha_emision')))
        ws.cell(row=fila, column=26, value=ref_info.get('codigo_tipo_doc', '01'))
        ws.cell(row=fila, column=27, value=ref_serie)
        ws.cell(row=fila, column=28, value=ref_numero)

    if factura.get('requiere_revision_tributaria', True):
        for col in range(1, 29):
            ws.cell(row=fila, column=col).fill = FILL_AMARILLO

print('escribir_fila_compra OK')

escribir_fila_compra OK


In [5]:
with open(INPUT_PATH, encoding='utf-8') as f:
    todas_facturas = json.load(f)

for f in todas_facturas:
    f.update(clasificar_tributario(f))

compras = sorted([f for f in todas_facturas if f.get('tipo_operacion') == 'COMPRA'],
                  key=lambda x: x.get('fecha_emision', ''))
lookup_ref = construir_lookup_referencias(todas_facturas)

todas_fechas = [f.get('fecha_emision', '') for f in todas_facturas if f.get('fecha_emision')]
periodo = datetime.now().strftime('%Y%m') if not todas_fechas else todas_fechas[0][:7].replace('-', '')

output_compras = OUTPUT_DIR / f'Registro_Compras_{periodo}.xlsx'
wb_compras = openpyxl.load_workbook(str(PLANTILLA_COMPRAS))
ws_compras = wb_compras.active

FILA_INICIO_COMPRAS = 14
ultimo_correlativo = get_ultimo_correlativo(ws_compras, FILA_INICIO_COMPRAS)
docs_existentes = leer_docs_existentes(ws_compras, FILA_INICIO_COMPRAS, col_serie=5, col_numero=7)

fila_actual = get_primera_fila_libre(ws_compras, FILA_INICIO_COMPRAS)
correlativo = ultimo_correlativo + 1
escritas = 0
ignoradas = 0

for f in compras:
    clave = f'{f.get("serie","")}-{f.get("numero","")}'
    if clave in docs_existentes:
        print(f'  Duplicado ignorado: {clave}')
        ignoradas += 1
        continue
    escribir_fila_compra(ws_compras, fila_actual, correlativo, f, lookup_ref)
    docs_existentes.add(clave)
    fila_actual += 1
    correlativo += 1
    escritas += 1

# RUC/razon social/periodo en el encabezado
ws_compras['B3'] = periodo
ws_compras['B4'] = CLIENTE.get('ruc')
ws_compras['B5'] = CLIENTE.get('razon_social')

wb_compras.save(str(output_compras))
print(f'Registro de Compras guardado: {output_compras}')
print(f'Filas escritas: {escritas}  Duplicados ignorados: {ignoradas}')

Registro de Compras guardado: ..\data\output\Registro_Compras_202502.xlsx
Filas escritas: 0  Duplicados ignorados: 0


## 4. Registro de Ventas (Formato 14.1)

In [6]:
def escribir_fila_venta(ws, fila: int, correlativo: int, factura: dict, lookup_ref: dict) -> None:
    tratamiento = factura.get('tratamiento_tributario')
    es_ajuste = factura.get('tipo_doc') in ('NOTA_CREDITO', 'NOTA_DEBITO')

    base = round2(factura.get('base_imponible'))
    igv = round2(factura.get('igv'))
    total = round2(factura.get('total'))
    # Normalizar: si extracción fue inconsistente, recalcular desde total con IGV 18%
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv = round2(total - base)

    ws.cell(row=fila, column=1, value=correlativo)
    ws.cell(row=fila, column=2, value=parse_fecha(factura.get('fecha_emision')))
    ws.cell(row=fila, column=3, value=parse_fecha(factura.get('fecha_vencimiento_pago')))
    ws.cell(row=fila, column=4, value=factura.get('codigo_tipo_doc'))
    ws.cell(row=fila, column=5, value=factura.get('serie'))
    ws.cell(row=fila, column=6, value=factura.get('numero'))
    ws.cell(row=fila, column=7, value=factura.get('tipo_doc_identidad_receptor'))
    numero_doc_id = factura.get('numero_doc_identidad_receptor') or factura.get('ruc_receptor')
    ws.cell(row=fila, column=8, value=numero_doc_id)
    ws.cell(row=fila, column=9, value=factura.get('razon_social_receptor'))

    if tratamiento == 'GRAVADA':
        ws.cell(row=fila, column=11, value=base)
        ws.cell(row=fila, column=15, value=igv)

    ws.cell(row=fila, column=17, value=total)

    if factura.get('moneda') == 'USD' and factura.get('tipo_cambio'):
        ws.cell(row=fila, column=18, value=factura.get('tipo_cambio'))

    if es_ajuste:
        ref_serie, ref_numero = parsear_referencia(factura.get('doc_referencia'))
        ref_info = lookup_ref.get((ref_serie, ref_numero), {})
        ws.cell(row=fila, column=19, value=parse_fecha(ref_info.get('fecha_emision')))
        ws.cell(row=fila, column=20, value=ref_info.get('codigo_tipo_doc', '01'))
        ws.cell(row=fila, column=21, value=ref_serie)
        ws.cell(row=fila, column=22, value=ref_numero)

    if factura.get('requiere_revision_tributaria', True):
        for col in range(1, 23):
            ws.cell(row=fila, column=col).fill = FILL_AMARILLO

print('escribir_fila_venta OK')

escribir_fila_venta OK


In [7]:
ventas = sorted([f for f in todas_facturas if f.get('tipo_operacion') == 'VENTA'],
                key=lambda x: x.get('fecha_emision', ''))

output_ventas = OUTPUT_DIR / f'Registro_Ventas_{periodo}.xlsx'
wb_ventas = openpyxl.load_workbook(str(PLANTILLA_VENTAS))
ws_ventas = wb_ventas.active

FILA_INICIO_VENTAS = 12
ultimo_correlativo_v = get_ultimo_correlativo(ws_ventas, FILA_INICIO_VENTAS)
docs_existentes_v = leer_docs_existentes(ws_ventas, FILA_INICIO_VENTAS, col_serie=5, col_numero=6)

fila_actual_v = get_primera_fila_libre(ws_ventas, FILA_INICIO_VENTAS)
correlativo_v = ultimo_correlativo_v + 1
escritas_v = 0
ignoradas_v = 0

for f in ventas:
    clave = f'{f.get("serie","")}-{f.get("numero","")}'
    if clave in docs_existentes_v:
        print(f'  Duplicado ignorado: {clave}')
        ignoradas_v += 1
        continue
    escribir_fila_venta(ws_ventas, fila_actual_v, correlativo_v, f, lookup_ref)
    docs_existentes_v.add(clave)
    fila_actual_v += 1
    correlativo_v += 1
    escritas_v += 1

ws_ventas['B3'] = periodo
ws_ventas['B4'] = CLIENTE.get('ruc')
ws_ventas['B5'] = CLIENTE.get('razon_social')

wb_ventas.save(str(output_ventas))
print(f'Registro de Ventas guardado: {output_ventas}')
print(f'Filas escritas: {escritas_v}  Duplicados ignorados: {ignoradas_v}')

Registro de Ventas guardado: ..\data\output\Registro_Ventas_202502.xlsx
Filas escritas: 3  Duplicados ignorados: 0


## 5. Verificación de totales

In [8]:
print('=== VERIFICACION DE TOTALES ===')

suma_compras_json = sum(round2(f.get('total')) for f in compras)
suma_ventas_json = sum(round2(f.get('total')) for f in ventas)

wb_check_c = openpyxl.load_workbook(str(output_compras))
ws_check_c = wb_check_c.active
suma_compras_excel = sum(
    round2(row[19]) for row in ws_check_c.iter_rows(min_row=FILA_INICIO_COMPRAS, values_only=True)
    if row[19] is not None
)

wb_check_v = openpyxl.load_workbook(str(output_ventas))
ws_check_v = wb_check_v.active
suma_ventas_excel = sum(
    round2(row[16]) for row in ws_check_v.iter_rows(min_row=FILA_INICIO_VENTAS, values_only=True)
    if row[16] is not None
)

print(f'Compras — JSON: {suma_compras_json:.2f}  Excel: {suma_compras_excel:.2f}  {"OK" if abs(suma_compras_json - suma_compras_excel) < 0.02 else "DIFERENCIA"}')
print(f'Ventas  — JSON: {suma_ventas_json:.2f}  Excel: {suma_ventas_excel:.2f}  {"OK" if abs(suma_ventas_json - suma_ventas_excel) < 0.02 else "DIFERENCIA"}')

n_revision = sum(1 for f in todas_facturas if f.get('requiere_revision_tributaria'))
print(f'Facturas marcadas para revision tributaria: {n_revision}')

=== VERIFICACION DE TOTALES ===
Compras — JSON: 0.00  Excel: 0.00  OK
Ventas  — JSON: 19470.00  Excel: 19470.00  OK
Facturas marcadas para revision tributaria: 1
